In [47]:
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os, sys, re

In [ ]:
adata = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/raw/single-cell/sc_matched.h5ad')
st_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/nsclc_cosmx/raw/spatial/st_concat.h5ad')

/home/jwinchel/miniconda3/envs/rctd/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [57]:
sys.path.append('/home/jwinchel/Syndri/Heimdall/')

from Heimdall.utils import convert_to_ensembl_ids

st_data = convert_to_ensembl_ids(st_data,'/work/magroup/skrieger/Syndri/data/')[0]

human
/work/magroup/skrieger/Syndri/data
Mapping data directory: /work/magroup/skrieger/Syndri/data/gene_mapping/ensembl/human
Loading mapping from cache: /work/magroup/skrieger/Syndri/data/gene_mapping/ensembl/human/symbol_to_ensembl.json
Successfully mapped 273 out of 280 identifiers (97.5%)


In [58]:
adata.var['gene_ensembl'] = adata.var.index

In [61]:
st_data.var

,gene_symbol,gene_ensembl
index,,
ENSG00000121270,ABCC11,ENSG00000121270
ENSG00000107796,ACTA2,ENSG00000107796
ENSG00000163017,ACTG2,ENSG00000163017
ENSG00000168615,ADAM9,ENSG00000168615
ENSG00000123146,ADGRE5,ENSG00000123146
...,...,...
ENSG00000110799,VWF,ENSG00000110799
WARS,WARS,N/A
ENSG00000148516,ZEB1,ENSG00000148516


In [62]:
# st_data.var_names
gene_overlap = list(set(st_data.var_names).intersection(set(adata.var_names)))
st_data = st_data[:,gene_overlap]
adata = adata[:,gene_overlap]
st_data.shape,adata.shape
# len(gene_overlap)

((1749131, 270), (10689, 270))

In [74]:
st_data.X.sum(axis=1).A.T.squeeze()

array([159., 226., 130., ..., 201., 342., 163.],
      shape=(1749131,), dtype=float32)

In [77]:
zero_valued = st_data.obs_names[st_data.X.sum(axis=1).A.T.squeeze()==0]

In [78]:
st_data[zero_valued].X

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [ ]:
adata.obs['Level3']

In [ ]:
st_data.obs['brain_section_label'].unique()

In [ ]:
sys.path.append('/home/jwinchel/Syndri/code/')
import importlib
import mapping
importlib.reload(mapping)
from mapping import integrate_single_st_file_with_harmony

slices = list(st_data.obs['brain_section_label'].unique())

st_data_list = []
for sl in slices:
    # print(sl,st_data.obs['slice'].unique(),len(st_data[st_data.obs['slice']==str(sl)]))
    adata,sl_adata, _ = integrate_single_st_file_with_harmony(st_data[st_data.obs['brain_section_label']==str(sl)], adata,tissue_filter=None,n_pcs=30,tissue_name=str(sl))
    st_data_list.append(sl_adata)



In [ ]:
adata.obs['cell_label']

In [ ]:
ad.settings.allow_write_nullable_strings = True
adata.write_h5ad('/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/interim/single-cell/sc_integrated.h5ad')

In [ ]:
st_data_list[0]

In [ ]:
list(st_data_list[0].obsm.keys())

In [ ]:
slices = list(st_data.obs['brain_section_label'].unique())
combined = np.full((st_data.n_obs,30),np.nan,dtype=adata.obsm['X_pca_harmony_Zhuang_ABCA_4_003'].dtype)

# map full adata cell IDs -> row indices
obs_to_idx = {
    obs_name: i
    for i, obs_name in enumerate(st_data.obs_names)
}

for i,subset in enumerate(st_data_list):
    idx = np.array([
        obs_to_idx[obs_name]
        for obs_name in subset.obs_names
    ])
    harmony_layer = [k for k in list(subset.obsm.keys()) if "X_pca_harmony" in k][0]
    combined[idx] = subset.obsm[harmony_layer]

st_data.obsm['X_pca_harmony'] = combined
st_data

In [ ]:
st_data

In [ ]:
slices

In [ ]:
for sl in slices:
    st_data[st_data.obs['brain_section_label']==sl].write_h5ad(f'/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/interim/spatial/st_integrated_slice-{sl}.h5ad')

In [ ]:
del st_data_list

In [ ]:
adata.obsm['X_pca_harmony'] = adata.obsm['X_pca_harmony_0'].copy()

In [ ]:
st_data[st_data.obs['brain_section_label']==slices[1]].obs#['class'].value_counts()

In [ ]:
adata

In [ ]:
# read in previously saved single-cell and spatial data
adata=ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/interim/single-cell/sc_integrated.h5ad')
st_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/interim/spatial/st_integrated_slice-Zhuang-ABCA-4.003.h5ad')
# st_data.obs[['center_x','center_y']] = st_data.obs[['center_x','center_y']].astype(float)
# st_data.obs[['min_x','min_y']] = st_data.obs[['min_x','min_y']].astype(float)
# st_data = st_data[st_data.obs['age']=='90wk']
adata.obsm['X_pca_harmony'] = adata.obsm['X_pca_harmony_Zhuang_ABCA_4_003'].copy()

In [ ]:
# # drop a few T-cells
# rng = np.random.default_rng(0)
# t_idx = np.flatnonzero(adata.obs["cell_type"].eq("T cell").to_numpy())
# drop_idx = rng.choice(t_idx, size=len(t_idx) - 91, replace=False)

# keep = np.ones(adata.n_obs, dtype=bool)
# keep[drop_idx] = False
# adata_subset = adata[keep].copy()

In [ ]:
adata

In [ ]:
st_data = st_data[~st_data.obs['class'].isna()]
st_data = st_data[~st_data.obs['cluster'].isin(['4808 PSV Pax2 Gly-Gaba_1', '5195 CBX Purkinje Gaba_2'])] # drop missing clusters from reference (presumably in 10Xv2)
st_data

In [ ]:
import sys
import numpy as np
sys.path.append('/home/jwinchel/Syndri/code')
from mapping import map_sc_reference_to_unique_xenium_cells

sc_mapped = map_sc_reference_to_unique_xenium_cells(
    # adata_subset,
    adata,
    st_data,
    embedding_key="X_pca_harmony",
    spatial_cols=("x", "y"),
    n_matches=50,
    reuse_penalty=9.0,
    rng=np.random.default_rng(0),
    max_reuse_per_sc=5000,
    match_cell_type=True,
    ensure_all_sc_used=False,
    sc_cell_type_col="cluster",
    st_cell_type_col="cluster",
)

In [ ]:
sc_mapped.uns['sc_reference_to_spatial_mapping']['spatial_cols'] = list(sc_mapped.uns['sc_reference_to_spatial_mapping']['spatial_cols'])
sc_mapped.obs['split'] = 'train'
splits = adata.obs.groupby('class').cumcount().apply(lambda x: x % 5)
val_ids = splits[splits==4].index.tolist()


sc_mapped.obs.loc[sc_mapped.obs['source_sc_id'].isin(val_ids),'split'] = 'val'

# cell type
cell_types = sorted(sc_mapped.obs['class'].unique())
cell_type_dict = {cell_types[i]: -(i+1) for i in range(len(cell_types))}
sc_mapped.uns['cell_type_dict'] = cell_type_dict
sc_mapped.obs['cell_type'] = sc_mapped.obs['class']
sc_mapped.write_h5ad('/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/interim/single-cell/sc_integrated_mapped-Zhuang-ABCA-4.003.h5ad')

In [ ]:
import seaborn as sns
st_data = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/brain_merfish/interim/spatial/st_integrated_slice-Zhuang-ABCA-4.003.h5ad')
# st_data = st_data[st_data.obs.sample(frac=0.1).index]
# st_data.obs[['center_x','center_y']] = st_data.obs[['center_x','center_y']].astype(float)
# st_data.obs[['min_x','min_y']] = st_data.obs[['min_x','min_y']].astype(float)

In [ ]:
st_data

In [ ]:
import seaborn as sns
ax=sns.scatterplot(
    x=st_data.obs['x'],
    y=st_data.obs['y'],
    hue=st_data.obs['class'],
    s=1,
    palette='tab20'
)
ax.legend(bbox_to_anchor=(1,1.05),markerscale=5)